# 03b · Download from CIVL's event portal (civlcomps.org)

Many competitions scored with FS publish their results on civlcomps.org, CIVL's event and ranking portal, and since 2024 organisers upload a zip of each task's IGC files there too. Most of these competitions are not on Flymaster, AirScore or Airtribune. Used with the permission of CIVL.

For every finished paragliding XC event from `FROM_YEAR` on, the script (`scripts/civlcomps.py`) saves into `civlcomps_igc/<event_id>/` in the project folder:

| File | Content |
|---|---|
| `results.html` | the event's results page (one row per task) |
| `participants.html` | registered pilots: CIVL ID, competition number, glider |
| `<task>.zip` | the task's IGC files |
| `<task>_results.html` | the task's FS results: pilots, turnpoints with coordinates, task statistics, GAP parameters |

and `civlcomps_catalog.csv` with one row per task. One request every 2 seconds. Files already downloaded are skipped, so **if Colab disconnects, Run All again** and it carries on. A sample of the portal suggests roughly 1,000 tasks with IGC from 2024 to 2026 (tens of thousands of tracks; some events are also on Airtribune or Flymaster, and the catalog step removes the duplicates).

## 1. Settings

In [ ]:
DRIVE_FOLDER_ID = "1_JO0XMMQ4NKVfpeA8uKQRkZZPNC4sqZ7"   # project folder in Google Drive (the id at the end of its link)
FROM_YEAR = 2023     # IGC uploads began in 2024; earlier years have results only
REPO_URL = "https://github.com/RobertojonesFAI/Pre-task-analyzer.git"
REPO = "/content/Pre-task-analyzer"

## 2. Mount Drive and get the code

In [ ]:
import os, shutil, subprocess
from google.colab import auth, drive
from googleapiclient.discovery import build

drive.mount("/content/drive")
auth.authenticate_user()
service = build("drive", "v3")
FIELDS = "id, name, parents, driveId, ownedByMe"


def path_up(file_id, root):
    """Folder names from a file up to the top of its drive, joined under root."""
    names, current = [], file_id
    while True:
        meta = service.files().get(fileId=current, fields=FIELDS, supportsAllDrives=True).execute()
        if not meta.get("parents"):
            return os.path.join(root, *reversed(names))
        names.append(meta["name"])
        current = meta["parents"][0]


def drive_path(folder_id):
    """A writable local path for a Drive folder, from its id."""
    meta = service.files().get(fileId=folder_id, fields=FIELDS, supportsAllDrives=True).execute()
    if meta.get("driveId"):  # in a shared drive
        drive_name = service.drives().get(driveId=meta["driveId"]).execute()["name"]
        return path_up(folder_id, f"/content/drive/Shareddrives/{drive_name}")
    if meta.get("ownedByMe"):  # in my own Drive
        return path_up(folder_id, "/content/drive/MyDrive")
    # Shared with me: write through a shortcut to it in my Drive
    page = None
    while True:
        res = service.files().list(
            q="mimeType='application/vnd.google-apps.shortcut' and trashed=false",
            fields="nextPageToken, files(id, shortcutDetails)", pageToken=page).execute()
        for f in res.get("files", []):
            if f.get("shortcutDetails", {}).get("targetId") == folder_id:
                return path_up(f["id"], "/content/drive/MyDrive")
        page = res.get("nextPageToken")
        if not page:
            raise SystemExit(f"'{meta['name']}' is shared with you. Add a shortcut to it in your Drive "
                             "(right click -> Organize -> Add shortcut), then rerun this cell.")


ROOT = drive_path(DRIVE_FOLDER_ID)
OUT = os.path.join(ROOT, "civlcomps_igc")
os.makedirs(OUT, exist_ok=True)
print("Downloads go to", OUT)

if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)


def run(*cmd):
    proc = subprocess.Popen(["python3", "-u", *cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end="")
    return proc.wait()

## 3. Download

Reads the events calendar year by year, then each event's results page (one request per event), then downloads the IGC zips and results pages. Prints one line per task.

In [ ]:
run(f"{REPO}/scripts/civlcomps.py", OUT, "--from-year", str(FROM_YEAR))

## 4. How many tracks

In [ ]:
import glob, zipfile
zips = glob.glob(os.path.join(OUT, "*", "*.zip"))
n = 0
for z in zips:
    try:
        n += sum(m.lower().endswith(".igc") and "__MACOSX" not in m for m in zipfile.ZipFile(z).namelist())
    except zipfile.BadZipFile:
        pass
print(f"{len(zips)} task packages, {n:,} IGC files")